In [ ]:
# Data manipulation and analysis
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine Learning
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

# Others
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("✅ All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

In [ ]:
# Load the dataset
df = pd.read_csv('Mobile Reviews Sentiment null.csv', encoding='latin1')

# Display basic information
print("="*60)
print("DATASET INFORMATION")
print("="*60)
print(f"\n📊 Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
print("📋 First 5 rows:")
df.head()

In [ ]:
# Check data types and missing values
print("="*60)
print("DATA TYPES AND MISSING VALUES")
print("="*60)
print("\n📋 Column Information:")
df.info()

print("\n\n📊 Missing Values Count:")
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
print(missing_values)

# Calculate missing percentage
print("\n📊 Missing Values Percentage:")
for col in missing_values.index:
    print(f"  {col}: {missing_values[col]/len(df)*100:.2f}%")

In [ ]:
# Statistical summary
print("="*60)
print("STATISTICAL SUMMARY")
print("="*60)
print("\n📊 Numerical Columns Statistics:")
df.describe()

In [ ]:
# Create a copy for cleaning
df_clean = df.copy()
initial_shape = df_clean.shape[0]

print("="*60)
print("DATA CLEANING PROCESS")
print("="*60)

# 1. Remove duplicates
df_clean = df_clean.drop_duplicates()
print(f"\n✅ Removed {initial_shape - df_clean.shape[0]} duplicate rows")

# 2. Handle missing values in numerical columns
num_cols = ['price_usd', 'rating', 'battery_life_rating', 'camera_rating', 
            'performance_rating', 'design_rating', 'display_rating', 'helpful_votes']

for col in num_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())
        print(f"✅ Filled missing values in '{col}' with median")

# 3. Handle missing values in categorical columns
if 'sentiment' in df_clean.columns:
    df_clean['sentiment'] = df_clean['sentiment'].fillna('Neutral')
    print("✅ Filled missing 'sentiment' with 'Neutral'")

# 4. Convert price_usd to numeric and drop NaN
df_clean['price_usd'] = pd.to_numeric(df_clean['price_usd'], errors='coerce')
df_clean = df_clean.dropna(subset=['price_usd'])
print(f"✅ Converted price_usd to numeric and cleaned invalid entries")

# 5. Drop rows without brand or model
df_clean = df_clean.dropna(subset=['brand', 'model'])

print(f"\n✅ Final dataset shape: {df_clean.shape}")
print(f"✅ Data after cleaning: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")

In [ ]:
# Feature Engineering
print("="*60)
print("FEATURE ENGINEERING")
print("="*60)

# 1. Create Overall Score (average of all ratings)
rating_cols = ['rating', 'battery_life_rating', 'camera_rating', 
               'performance_rating', 'design_rating', 'display_rating']
df_clean['overall_score'] = df_clean[rating_cols].mean(axis=1)
print("✅ Created 'overall_score' feature")

# 2. Create Price Segment
df_clean['price_segment'] = pd.cut(df_clean['price_usd'], 
                                   bins=[0, 200, 400, 600, 10000], 
                                   labels=['Budget', 'Mid-Range', 'Premium', 'Ultra-Premium'])
print("✅ Created 'price_segment' feature")

# 3. Create Age Group
df_clean['age_group'] = pd.cut(df_clean['age'], bins=[0, 25, 35, 50, 100], 
                               labels=['Young (18-25)', 'Adult (26-35)', 
                                       'Middle-Aged (36-50)', 'Senior (50+)'])
print("✅ Created 'age_group' feature")

# 4. Create Price to Rating Ratio
df_clean['price_per_rating'] = df_clean['price_usd'] / df_clean['rating']
print("✅ Created 'price_per_rating' feature")

# Display sample
print("\n📋 Sample of engineered features:")
display_cols = ['brand', 'model', 'price_usd', 'rating', 'overall_score', 
                'price_segment', 'age_group', 'price_per_rating']
df_clean[display_cols].head()

In [ ]:
# 1. Brand Distribution Analysis
print("="*60)
print("BRAND DISTRIBUTION ANALYSIS")
print("="*60)

brand_counts = df_clean['brand'].value_counts()
print(f"\n📊 Total unique brands: {df_clean['brand'].nunique()}")
print("\n🏷️ Top 10 Brands by Review Count:")
print(brand_counts.head(10))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar plot
brand_counts.head(10).plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('Top 10 Brands by Number of Reviews', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Brand')
axes[0].set_ylabel('Number of Reviews')
axes[0].tick_params(axis='x', rotation=45)

# Add value labels on bars
for i, v in enumerate(brand_counts.head(10).values):
    axes[0].text(i, v + 10, str(v), ha='center', fontsize=10)

# Pie chart
brand_counts.head(5).plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90, 
                          explode=[0.05, 0, 0, 0, 0])
axes[1].set_title('Top 5 Brands Market Share', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# 2. Rating Distribution and Sentiment Analysis
print("="*60)
print("RATING AND SENTIMENT ANALYSIS")
print("="*60)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Rating distribution
axes[0].hist(df_clean['rating'], bins=20, color='lightgreen', edgecolor='black', alpha=0.7)
axes[0].axvline(df_clean['rating'].mean(), color='red', linestyle='--', 
                label=f'Mean: {df_clean["rating"].mean():.2f}')
axes[0].axvline(df_clean['rating'].median(), color='blue', linestyle='--', 
                label=f'Median: {df_clean["rating"].median():.2f}')
axes[0].set_title('Distribution of User Ratings', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Rating (1-5)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Sentiment distribution
sentiment_counts = df_clean['sentiment'].value_counts()
colors = {'Positive': 'green', 'Neutral': 'orange', 'Negative': 'red'}
sentiment_counts.plot(kind='bar', ax=axes[1], color=[colors.get(x, 'gray') for x in sentiment_counts.index], 
                      edgecolor='black')
axes[1].set_title('Sentiment Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Sentiment')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

# Add value labels
for i, v in enumerate(sentiment_counts.values):
    axes[1].text(i, v + 50, str(v), ha='center', fontsize=10)

# Average rating by sentiment
avg_rating_by_sentiment = df_clean.groupby('sentiment')['rating'].mean()
avg_rating_by_sentiment.plot(kind='bar', ax=axes[2], color=[colors.get(x, 'gray') for x in avg_rating_by_sentiment.index], 
                              edgecolor='black')
axes[2].set_title('Average Rating by Sentiment', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Sentiment')
axes[2].set_ylabel('Average Rating')
axes[2].tick_params(axis='x', rotation=0)
axes[2].axhline(y=df_clean['rating'].mean(), color='black', linestyle='--', label='Overall Average')
axes[2].legend()

plt.tight_layout()
plt.show()

print("\n📊 Sentiment Statistics:")
print(sentiment_counts)
print(f"\nPositive Sentiment Percentage: {sentiment_counts['Positive']/len(df_clean)*100:.1f}%")
print(f"Negative Sentiment Percentage: {sentiment_counts['Negative']/len(df_clean)*100:.1f}%")
print("\n📊 Average Rating by Sentiment:")
print(avg_rating_by_sentiment)

In [ ]:
# 3. Price Analysis
print("="*60)
print("PRICE ANALYSIS")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Price distribution
axes[0].hist(df_clean['price_usd'], bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of Mobile Prices', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Price (USD)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df_clean['price_usd'].median(), color='red', linestyle='--', 
                linewidth=2, label=f'Median: ${df_clean["price_usd"].median():.0f}')
axes[0].axvline(df_clean['price_usd'].mean(), color='blue', linestyle='--', 
                linewidth=2, label=f'Mean: ${df_clean["price_usd"].mean():.0f}')
axes[0].legend()

# Price by segment
price_by_segment = df_clean.groupby('price_segment')['price_usd'].mean()
bars = axes[1].bar(price_by_segment.index, price_by_segment.values, 
                   color=['green', 'orange', 'blue', 'red'], edgecolor='black')
axes[1].set_title('Average Price by Segment', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Price Segment')
axes[1].set_ylabel('Average Price (USD)')
axes[1].tick_params(axis='x', rotation=45)

# Add value labels
for bar, value in zip(bars, price_by_segment.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
                 f'${value:.0f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print("\n💰 Price Statistics:")
print(f"  Minimum Price: ${df_clean['price_usd'].min():.2f}")
print(f"  Maximum Price: ${df_clean['price_usd'].max():.2f}")
print(f"  Average Price: ${df_clean['price_usd'].mean():.2f}")
print(f"  Median Price: ${df_clean['price_usd'].median():.2f}")
print(f"  Standard Deviation: ${df_clean['price_usd'].std():.2f}")

print("\n📊 Price Segment Distribution:")
segment_dist = df_clean['price_segment'].value_counts()
for segment in ['Budget', 'Mid-Range', 'Premium', 'Ultra-Premium']:
    if segment in segment_dist.index:
        print(f"  {segment}: {segment_dist[segment]} products ({segment_dist[segment]/len(df_clean)*100:.1f}%)")

In [ ]:
# 4. Price vs Rating Analysis
print("="*60)
print("PRICE VS RATING ANALYSIS")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter plot
scatter = axes[0].scatter(df_clean['price_usd'], df_clean['rating'], 
                          c=df_clean['overall_score'], cmap='viridis', alpha=0.6, s=30)
axes[0].set_xlabel('Price (USD)')
axes[0].set_ylabel('Rating')
axes[0].set_title('Price vs Rating (Color: Overall Score)', fontsize=12, fontweight='bold')
plt.colorbar(scatter, ax=axes[0], label='Overall Score')

# Add trend line
z = np.polyfit(df_clean['price_usd'], df_clean['rating'], 1)
p = np.poly1d(z)
axes[0].plot(df_clean['price_usd'].sort_values(), p(df_clean['price_usd'].sort_values()), 
            "r--", linewidth=2, label=f'Trend line (slope: {z[0]:.4f})')
axes[0].legend()

# Box plot by price segment
df_clean.boxplot(column='rating', by='price_segment', ax=axes[1])
axes[1].set_title('Rating Distribution by Price Segment', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Price Segment')
axes[1].set_ylabel('Rating')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('')
plt.tight_layout()
plt.show()

# Correlation analysis
correlation = df_clean['price_usd'].corr(df_clean['rating'])
print(f"\n📈 Correlation between Price and Rating: {correlation:.3f}")
if correlation > 0:
    print("   → Positive correlation: Higher priced phones tend to have slightly better ratings")
else:
    print("   → Negative correlation: Lower priced phones have better ratings")

# Price to rating ratio by segment
print("\n💰 Price-to-Rating Ratio by Segment:")
price_ratio = df_clean.groupby('price_segment')['price_per_rating'].mean().round(2)
for segment, ratio in price_ratio.items():
    print(f"  {segment}: ${ratio:.2f} per rating point")

In [ ]:
# 5. Correlation Heatmap
print("="*60)
print("CORRELATION ANALYSIS")
print("="*60)

# Select numerical columns
numeric_cols = ['price_usd', 'rating', 'overall_score', 'battery_life_rating', 
                'camera_rating', 'performance_rating', 'design_rating', 'display_rating',
                'helpful_votes', 'age']

corr_matrix = df_clean[numeric_cols].corr()

# Create heatmap
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', square=True, linewidths=1, mask=mask)
plt.title('Correlation Heatmap of Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Key insights
print("\n🔍 Key Correlation Insights:")
print(f"  • Rating vs Overall Score: {corr_matrix.loc['rating', 'overall_score']:.3f} (Strong correlation)")
print(f"  • Price vs Rating: {corr_matrix.loc['price_usd', 'rating']:.3f} (Weak correlation)")
print(f"  • Performance vs Rating: {corr_matrix.loc['performance_rating', 'rating']:.3f}")
print(f"  • Camera vs Rating: {corr_matrix.loc['camera_rating', 'rating']:.3f}")
print(f"  • Battery vs Rating: {corr_matrix.loc['battery_life_rating', 'rating']:.3f}")
print(f"  • Design vs Rating: {corr_matrix.loc['design_rating', 'rating']:.3f}")
print(f"  • Display vs Rating: {corr_matrix.loc['display_rating', 'rating']:.3f}")

In [ ]:
# Select features for clustering
print("="*60)
print("FEATURE PREPARATION FOR CLUSTERING")
print("="*60)

cluster_features = ['price_usd', 'rating', 'overall_score', 'battery_life_rating', 
                    'camera_rating', 'performance_rating', 'design_rating', 'display_rating']

# Create feature matrix
X = df_clean[cluster_features].copy()

# Handle any remaining missing values
X = X.fillna(X.median())

# Add brand encoding
le = LabelEncoder()
df_clean['brand_encoded'] = le.fit_transform(df_clean['brand'])
X['brand_encoded'] = df_clean['brand_encoded']

# Add price segment encoding
df_clean['price_segment_encoded'] = df_clean['price_segment'].astype('category').cat.codes
X['price_segment_encoded'] = df_clean['price_segment_encoded']

print("\n📊 Features selected for clustering:")
for i, feature in enumerate(X.columns, 1):
    print(f"  {i}. {feature}")
print(f"\n📋 Feature matrix shape: {X.shape}")

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print("\n✅ Features scaled successfully!")
print(f"📊 Scaled data shape: {X_scaled_df.shape}")
print("\n📋 Scaled features statistics:")
print(X_scaled_df.describe().round(3))

In [ ]:
# Determine optimal number of clusters using Elbow Method
print("="*60)
print("OPTIMAL CLUSTERS DETERMINATION")
print("="*60)

inertias = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))
    print(f"  k={k}: Inertia={kmeans.inertia_:.0f}, Silhouette={silhouette_scores[-1]:.3f}")

# Plot Elbow curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[0].set_ylabel('Inertia', fontsize=12)
axes[0].set_title('Elbow Method for Optimal k', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=4, color='red', linestyle='--', label='Selected k=4')
axes[0].legend()

axes[1].plot(K_range, silhouette_scores, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score for Different k', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].axvline(x=4, color='red', linestyle='--', label='Selected k=4')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n📊 Silhouette Scores Summary:")
for k, score in zip(K_range, silhouette_scores):
    print(f"  k={k}: {score:.3f}")
print("\n✅ Based on elbow method and silhouette score, k=4 is optimal for our segmentation.")

In [ ]:
# Perform K-Means clustering with k=4
print("="*60)
print("K-MEANS CLUSTERING (k=4)")
print("="*60)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_clean['cluster'] = kmeans.fit_predict(X_scaled)

print(f"\n✅ Clustering completed!")
print(f"\n📊 Cluster distribution:")
cluster_counts = df_clean['cluster'].value_counts().sort_index()
for cluster in sorted(df_clean['cluster'].unique()):
    print(f"  Cluster {cluster}: {cluster_counts[cluster]} products ({cluster_counts[cluster]/len(df_clean)*100:.1f}%)")

# Silhouette score for final model
final_silhouette = silhouette_score(X_scaled, df_clean['cluster'])
print(f"\n📈 Final Silhouette Score: {final_silhouette:.3f}")
if final_silhouette > 0.5:
    print("   → Good cluster separation")
elif final_silhouette > 0.3:
    print("   → Moderate cluster separation")
else:
    print("   → Weak cluster separation")

In [ ]:
# Cluster Analysis and Interpretation
print("="*60)
print("CLUSTER ANALYSIS AND INTERPRETATION")
print("="*60)

# Calculate cluster centers
cluster_centers = pd.DataFrame(kmeans.cluster_centers_, columns=X.columns)
cluster_centers.index.name = 'Cluster'

print("\n📊 Cluster Centers (Scaled Values):")
print(cluster_centers.round(3))

# Analyze each cluster
cluster_summary = df_clean.groupby('cluster').agg({
    'price_usd': ['mean', 'min', 'max', 'std'],
    'rating': 'mean',
    'overall_score': 'mean',
    'battery_life_rating': 'mean',
    'camera_rating': 'mean',
    'performance_rating': 'mean',
    'design_rating': 'mean',
    'display_rating': 'mean',
    'brand': lambda x: x.mode()[0] if len(x) > 0 else 'N/A',
    'model': 'count'
}).round(2)

print("\n📊 Cluster Summary Statistics:")
print(cluster_summary)

# Rename clusters based on analysis
cluster_names = {
    0: 'Budget Value',
    1: 'Mid-Range Balanced',
    2: 'Premium Performance',
    3: 'Ultra-Premium Luxury'
}
df_clean['cluster_name'] = df_clean['cluster'].map(cluster_names)

print("\n🏷️ Cluster Interpretation:")
for cluster in sorted(df_clean['cluster'].unique()):
    cluster_data = df_clean[df_clean['cluster'] == cluster]
    print(f"\n  {'='*50}")
    print(f"  {cluster_names[cluster]} (Cluster {cluster}):")
    print(f"  {'='*50}")
    print(f"    • Average Price: ${cluster_data['price_usd'].mean():.2f}")
    print(f"    • Price Range: ${cluster_data['price_usd'].min():.2f} - ${cluster_data['price_usd'].max():.2f}")
    print(f"    • Average Rating: {cluster_data['rating'].mean():.2f}/5.0")
    print(f"    • Overall Score: {cluster_data['overall_score'].mean():.2f}/5.0")
    print(f"    • Battery Rating: {cluster_data['battery_life_rating'].mean():.2f}/5.0")
    print(f"    • Camera Rating: {cluster_data['camera_rating'].mean():.2f}/5.0")
    print(f"    • Performance Rating: {cluster_data['performance_rating'].mean():.2f}/5.0")
    print(f"    • Top Brand: {cluster_data['brand'].mode()[0]}")
    print(f"    • Number of Products: {len(cluster_data)}")
    print(f"    • Key Characteristics: ", end="")
    if cluster == 0:
        print("Lowest price, moderate ratings, best value for money")
    elif cluster == 1:
        print("Balanced price-performance, good all-rounder")
    elif cluster == 2:
        print("High-end features, excellent performance, premium pricing")
    else:
        print("Ultra-premium, top-tier specifications, luxury segment")

In [ ]:
# Visualize Clusters using PCA
print("="*60)
print("CLUSTER VISUALIZATION")
print("="*60)

# Apply PCA for 2D visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df_clean['pca_x'] = X_pca[:, 0]
df_clean['pca_y'] = X_pca[:, 1]

# Create scatter plot
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# By cluster
scatter1 = axes[0].scatter(df_clean['pca_x'], df_clean['pca_y'], 
                           c=df_clean['cluster'], cmap='viridis', alpha=0.6, s=50)
axes[0].set_xlabel(f'PCA Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PCA Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].set_title('Product Segments Visualization (PCA)', fontsize=12, fontweight='bold')
cbar = plt.colorbar(scatter1, ax=axes[0])
cbar.set_label('Cluster')

# By price segment
price_segment_colors = {'Budget': 'green', 'Mid-Range': 'orange', 
                        'Premium': 'blue', 'Ultra-Premium': 'red'}
for segment, color in price_segment_colors.items():
    segment_data = df_clean[df_clean['price_segment'] == segment]
    axes[1].scatter(segment_data['pca_x'], segment_data['pca_y'], 
                   c=color, label=segment, alpha=0.6, s=50)
axes[1].set_xlabel(f'PCA Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PCA Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].set_title('Price Segments Visualization', fontsize=12, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

# PCA explained variance
print(f"\n📊 PCA Explained Variance:")
print(f"  Component 1: {pca.explained_variance_ratio_[0]:.2%}")
print(f"  Component 2: {pca.explained_variance_ratio_[1]:.2%}")
print(f"  Total variance captured: {pca.explained_variance_ratio_.sum():.2%}")

In [ ]:
# Build Recommendation System using Cosine Similarity
print("="*60)
print("BUILDING RECOMMENDATION SYSTEM")
print("="*60)

# Select features for similarity calculation
features_for_sim = ['price_usd', 'rating', 'overall_score', 'battery_life_rating', 
                    'camera_rating', 'performance_rating', 'design_rating', 'display_rating']

# Create feature matrix for recommendations
rec_features = df_clean[features_for_sim].copy()
rec_features = rec_features.fillna(rec_features.median())

# Scale features
scaler_rec = StandardScaler()
rec_features_scaled = scaler_rec.fit_transform(rec_features)

# Compute cosine similarity matrix
similarity_matrix = cosine_similarity(rec_features_scaled)

# Create mapping from model to index
model_to_idx = {model: idx for idx, model in enumerate(df_clean['model'])}

print(f"✅ Similarity matrix shape: {similarity_matrix.shape}")
print(f"✅ Total products in recommendation database: {len(model_to_idx)}")

In [ ]:
# Function to get recommendations
def recommend_products(model_name, df, similarity_matrix, model_to_idx, top_n=5):
    """Get top N similar products for a given model."""
    
    if model_name not in model_to_idx:
        return pd.DataFrame()
    
    idx = model_to_idx[model_name]
    sim_scores = list(enumerate(similarity_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]  # Exclude the product itself
    
    product_indices = [i[0] for i in sim_scores]
    recommendations = df.iloc[product_indices][['brand', 'model', 'price_usd', 
                                                'rating', 'overall_score', 'cluster_name']].copy()
    recommendations['similarity_score'] = [i[1] for i in sim_scores]
    
    return recommendations

# Test the recommendation system
print("\n🔍 Testing Recommendation System:")
print("-"*40)

# Get a sample product
sample_model = df_clean['model'].iloc[0]
sample_brand = df_clean[df_clean['model'] == sample_model]['brand'].iloc[0]
print(f"\n📱 Selected Product: {sample_brand} {sample_model}")
print(f"   Price: ${df_clean[df_clean['model'] == sample_model]['price_usd'].iloc[0]:.2f}")
print(f"   Rating: {df_clean[df_clean['model'] == sample_model]['rating'].iloc[0]:.1f}/5.0")

# Get recommendations
recommendations = recommend_products(sample_model, df_clean, similarity_matrix, model_to_idx, top_n=5)

if not recommendations.empty:
    print(f"\n✅ Top 5 Similar Products:")
    print(recommendations.to_string(index=False))
else:
    print("\n❌ No recommendations found for this model.")

In [ ]:
# Visualize recommendations
if not recommendations.empty:
    # Get original product details
    original = df_clean[df_clean['model'] == sample_model].iloc[0]
    
    # Create comparison dataframe
    comparison_data = {
        'Product': [sample_model] + recommendations['model'].tolist(),
        'Price': [original['price_usd']] + recommendations['price_usd'].tolist(),
        'Rating': [original['rating']] + recommendations['rating'].tolist(),
        'Overall Score': [original['overall_score']] + recommendations['overall_score'].tolist()
    }
    comparison_df = pd.DataFrame(comparison_data)
    
    # Create visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Price comparison
    axes[0].barh(comparison_df['Product'], comparison_df['Price'], color='skyblue')
    axes[0].set_xlabel('Price (USD)')
    axes[0].set_title('Price Comparison', fontsize=12, fontweight='bold')
    
    # Rating comparison
    axes[1].barh(comparison_df['Product'], comparison_df['Rating'], color='lightgreen')
    axes[1].set_xlabel('Rating')
    axes[1].set_title('Rating Comparison', fontsize=12, fontweight='bold')
    
    # Overall Score comparison
    axes[2].barh(comparison_df['Product'], comparison_df['Overall Score'], color='coral')
    axes[2].set_xlabel('Overall Score')
    axes[2].set_title('Overall Score Comparison', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Recommendation Analysis:")
    print(f"  Original Product Rating: {original['rating']:.1f}/5.0")
    print(f"  Average Recommendation Rating: {recommendations['rating'].mean():.1f}/5.0")
    print(f"  Average Similarity Score: {recommendations['similarity_score'].mean():.3f}")
else:
    print("No recommendations to visualize.")

In [ ]:
# Business Insights
print("="*60)
print("BUSINESS INSIGHTS AND RECOMMENDATIONS")
print("="*60)

print("\n💡 Key Findings from Analysis:")
print("-"*40)

print("\n1. MARKET SEGMENTATION INSIGHTS:")
for cluster in sorted(df_clean['cluster'].unique()):
    cluster_data = df_clean[df_clean['cluster'] == cluster]
    print(f"\n   {cluster_names[cluster]} (Cluster {cluster}):")
    print(f"   - Market Share: {len(cluster_data)/len(df_clean)*100:.1f}%")
    print(f"   - Average Price: ${cluster_data['price_usd'].mean():.2f}")
    print(f"   - Customer Sentiment: {cluster_data['sentiment'].value_counts().index[0]}")

print("\n2. PRICE-PERFORMANCE INSIGHTS:")
print(f"   - Best Value Segment: {df_clean.groupby('cluster')['price_per_rating'].mean().idxmin()}")
print(f"   - Premium Segment: Cluster with highest average price")
print(f"   - Budget Segment: Largest market share segment")

print("\n3. RECOMMENDATIONS:")
print("   a) For Budget Segment:")
print("      - Focus on cost-effective marketing")
print("      - Highlight battery life and basic features")
print("      - Bundle deals and discounts")
print("\n   b) For Mid-Range Segment:")
print("      - Balance between price and features")
print("      - Target young professionals and students")
print("      - Improve camera and performance ratings")
print("\n   c) For Premium Segment:")
print("      - Focus on brand loyalty programs")
print("      - Highlight premium features and build quality")
print("      - Offer exclusive accessories")
print("\n   d) For Ultra-Premium Segment:")
print("      - Focus on luxury branding")
print("      - Offer premium customer service")
print("      - Highlight cutting-edge technology")

print("\n4. RECOMMENDATION SYSTEM VALUE:")
print("   - Helps customers find similar products based on their preferences")
print("   - Increases cross-selling opportunities")
print("   - Improves customer experience and satisfaction")

print("\n✅ Project Completed Successfully!")